# Vietnamese Currency Detection — Train and Evaluate

Notebook khung cho Google Colab. Notebook không tự động sửa dataset Roboflow. Việc tải dữ liệu và huấn luyện chỉ chạy sau khi điền dataset version và bật cờ thực thi.

## 1. Chuẩn bị runtime

Chọn `Runtime > Change runtime type > GPU`, sau đó chạy các cell theo thứ tự.

In [ ]:
!nvidia-smi
%pip install -q "ultralytics>=8.3,<9" "roboflow>=1.1,<2" pandas pyyaml scikit-learn seaborn

In [ ]:
from pathlib import Path
import json
import os
import platform
import torch
import ultralytics

assert torch.cuda.is_available(), "GPU chưa được bật trong Colab runtime."
print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "ultralytics": ultralytics.__version__,
})

## 2. Mount Google Drive và đọc secret

Tạo secret `ROBOFLOW_API_KEY` trong Colab Secrets. Không in giá trị secret ra output.

In [ ]:
from google.colab import drive, userdata

drive.mount('/content/drive')
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
assert ROBOFLOW_API_KEY, "Thiếu ROBOFLOW_API_KEY trong Colab Secrets."

DRIVE_ROOT = Path('/content/drive/MyDrive/nhan_dang_mau')
for directory in ['datasets', 'checkpoints/baseline', 'checkpoints/improved', 'runs', 'exports']:
    (DRIVE_ROOT / directory).mkdir(parents=True, exist_ok=True)

print(f'Artifacts sẽ được lưu tại: {DRIVE_ROOT}')

## 3. Cấu hình thí nghiệm

Version baseline đã được xác nhận là v1. Các cờ tải dữ liệu, smoke test và huấn luyện đầy đủ mặc định đều tắt để tránh chạy ngoài ý muốn.

In [ ]:
WORKSPACE = 'vnm-money-currency'
PROJECT = 'vietnamese-currency-lgi9i-4z7kn'
BASELINE_VERSION = 1
IMPROVED_VERSION = None  # TODO: điền sau khi tạo version cải tiến.

RUN_DOWNLOAD = False
RUN_SMOKE_TEST = False
RUN_FULL_TRAINING = False
RUN_BEST_VALIDATION = False
EXPERIMENT = 'baseline'  # 'baseline' hoặc 'improved'

assert not (RUN_SMOKE_TEST and RUN_FULL_TRAINING), 'Chỉ được bật một chế độ train.'
EXPECTED_CLASSES = ['1000', '10000', '100000', '2000', '20000', '200000', '5000', '50000', '500000']
EXPECTED_SPLITS = {'train': 1537, 'valid': 439, 'test': 218}

TRAIN_CONFIG = {
    'model': 'yolov8n.pt',
    'imgsz': 640,
    'epochs': 100,
    'batch': 16,
    'patience': 20,
    'seed': 42,
    'device': 0,
    'workers': 2,
}

## 4. Tải dataset version từ Roboflow

Dataset được tải vào `/content/datasets` để tránh đọc hàng nghìn file nhỏ trực tiếp từ Drive.

In [ ]:
from roboflow import Roboflow

version_number = BASELINE_VERSION if EXPERIMENT == 'baseline' else IMPROVED_VERSION
DATASET_ROOT = Path('/content/datasets') / EXPERIMENT

DATA_YAML = DATASET_ROOT / 'data.yaml'
if RUN_DOWNLOAD:
    assert version_number is not None, 'Chưa điền dataset version number.'
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(WORKSPACE).project(PROJECT)
    dataset = project.version(version_number).download('yolov8', location=str(DATASET_ROOT), overwrite=True)
    DATASET_ROOT = Path(dataset.location)
    DATA_YAML = DATASET_ROOT / 'data.yaml'
    assert DATA_YAML.exists(), f'Không tìm thấy {DATA_YAML}'
    print(DATA_YAML)
elif DATA_YAML.exists():
    print(f'Sử dụng dataset đã tải: {DATA_YAML}')
else:
    print('Chưa tải dataset: RUN_DOWNLOAD=False và data.yaml chưa tồn tại.')

## 5. Xác minh dataset trước khi train

Cell này dừng pipeline nếu class order hoặc số ảnh trong split không đúng baseline version 1.

In [ ]:
import yaml

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.avif'}
DATASET_VALIDATED = False

if DATA_YAML.exists():
    data_config = yaml.safe_load(DATA_YAML.read_text(encoding='utf-8'))
    names = data_config['names']
    if isinstance(names, dict):
        names = [names[key] for key in sorted(names, key=lambda value: int(value))]
    names = [str(name) for name in names]
    assert names == EXPECTED_CLASSES, f'Class mapping không đúng: {names}'

    actual_splits = {}
    for split in ('train', 'valid', 'test'):
        image_dir = DATASET_ROOT / split / 'images'
        assert image_dir.is_dir(), f'Thiếu thư mục {image_dir}'
        actual_splits[split] = sum(
            1 for path in image_dir.rglob('*')
            if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
        )
    assert actual_splits == EXPECTED_SPLITS, f'Split không đúng: {actual_splits}'
    DATASET_VALIDATED = True
    print({'classes': names, 'splits': actual_splits, 'validated': True})
else:
    print('Bỏ qua validation vì dataset chưa được tải.')

## 6. Smoke test hoặc huấn luyện đầy đủ

Bật `RUN_SMOKE_TEST=True` để chạy 1 epoch kiểm tra pipeline. Chỉ bật `RUN_FULL_TRAINING=True` sau khi smoke test thành công. Baseline tắt augmentation ở trainer.

In [ ]:
from ultralytics import YOLO

NO_TRAINER_AUG = {
    'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0,
    'degrees': 0.0, 'translate': 0.0, 'scale': 0.0,
    'shear': 0.0, 'perspective': 0.0,
    'flipud': 0.0, 'fliplr': 0.0,
    'mosaic': 0.0, 'mixup': 0.0, 'copy_paste': 0.0,
}

RUN_REQUESTED = RUN_SMOKE_TEST or RUN_FULL_TRAINING
if RUN_REQUESTED:
    assert DATASET_VALIDATED, 'Dataset phải được xác minh trước khi huấn luyện.'
    run_name = f'{EXPERIMENT}-smoke' if RUN_SMOKE_TEST else EXPERIMENT
    train_config = dict(TRAIN_CONFIG)
    if RUN_SMOKE_TEST:
        train_config.update({'epochs': 1, 'patience': 1})

    runtime_info = {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'cuda': torch.version.cuda,
        'gpu': torch.cuda.get_device_name(0),
        'ultralytics': ultralytics.__version__,
        'dataset_version': version_number,
        'experiment': EXPERIMENT,
        'smoke_test': RUN_SMOKE_TEST,
        'train_config': train_config,
    }
    metadata_path = DRIVE_ROOT / 'runs' / f'{run_name}-environment.json'
    metadata_path.write_text(json.dumps(runtime_info, ensure_ascii=False, indent=2), encoding='utf-8')

    model = YOLO(TRAIN_CONFIG['model'])
    results = model.train(
        data=str(DATA_YAML),
        project=str(DRIVE_ROOT / 'runs'),
        name=run_name,
        exist_ok=False,
        **{k: v for k, v in train_config.items() if k != 'model'},
        **NO_TRAINER_AUG,
    )
else:
    print('Chưa huấn luyện: cả RUN_SMOKE_TEST và RUN_FULL_TRAINING đều False.')

## 7. Xác minh checkpoint tốt nhất trên validation

Sau khi huấn luyện xong, đặt `RUN_BEST_VALIDATION=True` rồi chạy cell dưới. Bước này chỉ đọc `best.pt` và validation split, không thay đổi dataset và chưa sử dụng test set.

In [ ]:
if RUN_BEST_VALIDATION:
    assert DATASET_VALIDATED, 'Dataset phải được xác minh trước khi validation.'
    best_path = DRIVE_ROOT / 'runs' / EXPERIMENT / 'weights' / 'best.pt'
    assert best_path.is_file(), f'Không tìm thấy checkpoint: {best_path}'

    best_model = YOLO(str(best_path))
    best_metrics = best_model.val(
        data=str(DATA_YAML),
        split='val',
        imgsz=TRAIN_CONFIG['imgsz'],
        batch=TRAIN_CONFIG['batch'],
        device=TRAIN_CONFIG['device'],
        plots=True,
        project=str(DRIVE_ROOT / 'runs'),
        name=f'{EXPERIMENT}-best-val',
        exist_ok=True,
    )
    p = float(best_metrics.box.mp)
    r = float(best_metrics.box.mr)
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    print({
        'checkpoint': str(best_path),
        'precision': p,
        'recall': r,
        'f1': f1,
        'map50': float(best_metrics.box.map50),
        'map50_95': float(best_metrics.box.map),
        'speed_ms_per_image': best_metrics.speed,
    })
else:
    print('Chưa validation best.pt: RUN_BEST_VALIDATION=False.')

## 8. Các bước tiếp theo

- Lưu metric validation và artifact của `best.pt`.
- Quét confidence threshold trên validation.
- Khóa threshold.
- Đánh giá test đúng một lần với threshold đã khóa.
- Tính Exact Count Accuracy, Exact Total Accuracy, MAE tổng tiền và thời gian xử lý.
- Xuất confusion matrix, metric theo class và ảnh phân tích lỗi.